# 14 Interactive Content-Based Recommender

This notebook prepares the recommendation logic for the final interactive demo.

It uses the optimized hybrid feature representation selected in Notebook 13:

- Spotify audio features
- lyrics features
- PCA-compressed extracted audio features

The goal is to simulate how a user can interact with the recommender by liking and disliking songs. Based on this feedback, the system builds a temporary user taste profile and generates personalized recommendations.

In [1]:
from pathlib import Path
import sys
import json
import ast
import warnings

import numpy as np
import pandas as pd
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

processed_dir = PROJECT_ROOT / "data" / "processed"
models_dir = processed_dir / "models"

print("Project root:", PROJECT_ROOT)
print("Models dir:", models_dir)

Project root: c:\Users\hp\OneDrive - WU Wien\Desktop\Denis_Masters\Semester 2\Solution Engineering - Python\Spotify_Project\spotify_project
Models dir: c:\Users\hp\OneDrive - WU Wien\Desktop\Denis_Masters\Semester 2\Solution Engineering - Python\Spotify_Project\spotify_project\data\processed\models


In [2]:
tracks = pd.read_parquet(processed_dir / "tracks_with_predicted_genres.parquet")
audio_features = pd.read_parquet(processed_dir / "audio_features_clean.parquet")
lyrics_features = pd.read_parquet(processed_dir / "lyrics_features_valid_clean.parquet")
albums = pd.read_parquet(processed_dir / "albums_clean.parquet")
artists = pd.read_parquet(processed_dir / "artists_clean.parquet")

with open(models_dir / "optimized_recommender_config.json", "r") as f:
    config = json.load(f)

pca10_audio_features = joblib.load(models_dir / "pca10_audio_features.joblib")

print("Loaded config:")
config

Loaded config:


{'model_name': 'weighted_hybrid_recommender',
 'spotify_weight': 0.4,
 'lyrics_weight': 0.4,
 'pca_audio_weight': 0.2,
 'pca_components': 10,
 'spotify_features': ['danceability',
  'energy',
  'valence',
  'acousticness',
  'instrumentalness',
  'speechiness',
  'liveness',
  'tempo',
  'loudness',
  'popularity'],
 'lyrics_features': ['mean_syllables_word',
  'mean_words_sentence',
  'n_sentences',
  'n_words',
  'sentence_similarity',
  'vocabulary_wealth'],
 'extracted_audio_features': ['Chroma_1',
  'Chroma_10',
  'Chroma_11',
  'Chroma_12',
  'Chroma_2',
  'Chroma_3',
  'Chroma_4',
  'Chroma_5',
  'Chroma_6',
  'Chroma_7',
  'Chroma_8',
  'Chroma_9',
  'MEL_1',
  'MEL_10',
  'MEL_100',
  'MEL_101',
  'MEL_102',
  'MEL_103',
  'MEL_104',
  'MEL_105',
  'MEL_106',
  'MEL_107',
  'MEL_108',
  'MEL_109',
  'MEL_11',
  'MEL_110',
  'MEL_111',
  'MEL_112',
  'MEL_113',
  'MEL_114',
  'MEL_115',
  'MEL_116',
  'MEL_117',
  'MEL_118',
  'MEL_119',
  'MEL_12',
  'MEL_120',
  'MEL_121',
  

## Building the Recommendation Dataset

The interactive recommender needs one shared dataset that contains all information required for search, display, and recommendation:

- track metadata
- predicted genres
- Spotify audio features
- lyrics features
- PCA-compressed extracted audio features

Only tracks that have all required feature groups are kept, because the optimized hybrid model depends on all three components.

In [3]:
def to_list(val):
    if isinstance(val, list):
        return val
    if isinstance(val, np.ndarray):
        return val.tolist()
    if pd.isna(val):
        return []
    if isinstance(val, str):
        try:
            return ast.literal_eval(val)
        except:
            return [val]
    return []

In [4]:
recommender_df = (
    tracks
    .merge(
        audio_features,
        left_on="id",
        right_on="track_id",
        how="inner"
    )
    .merge(
        lyrics_features,
        left_on="id",
        right_on="track_id",
        how="inner",
        suffixes=("", "_lyrics")
    )
)

duplicate_track_id_cols = [
    col for col in recommender_df.columns
    if "track_id" in col and col != "track_id"
]

recommender_df = recommender_df.drop(
    columns=duplicate_track_id_cols,
    errors="ignore"
)

recommender_df["eval_genres"] = recommender_df["predicted_genre_list"].apply(to_list)

recommender_df = recommender_df[
    recommender_df["eval_genres"].str.len() > 0
].reset_index(drop=True)

print(f"Recommendation dataset rows: {len(recommender_df):,}")
print(f"Recommendation dataset columns: {len(recommender_df.columns):,}")

recommender_df[["id", "name", "popularity", "eval_genres"]].head()

Recommendation dataset rows: 71,405
Recommendation dataset columns: 248


,id,name,popularity,eval_genres
0,4PrAZpH9Ic7S47E78BN6E4,Already Gone,45.0,"[rock, pop, other]"
1,01zME4q62SDPtD0hOSmTrG,Creature Kind,47.0,"[rnb_soul, pop, other, electronic]"
2,2D0the6JBRyOpMeBkPFkGY,Greatest Comedian,36.0,[folk_acoustic]
3,0tewjlNbotxqF2obibsg36,I Wish I Was A Shark,32.0,[rock]
4,2Dh5wED4UVeiBqneUdc5Gy,Cola Falls,35.0,"[rock, other]"


In [5]:
pca10_audio_features = pca10_audio_features.loc[
    recommender_df.index
].reset_index(drop=True)

print("PCA Audio shape:", pca10_audio_features.shape)
print("Recommender rows:", len(recommender_df))

PCA Audio shape: (71405, 10)
Recommender rows: 71405


## Building the Optimized Hybrid Feature Matrix

Notebook 13 selected a weighted hybrid representation using:

- Spotify features: 40%
- lyrics features: 40%
- PCA audio features: 20%

Each feature group is standardized separately before weighting. This keeps the contribution of each group controlled and avoids one feature group dominating because of different numerical scales.

In [6]:
spotify_features = config["spotify_features"]
lyrics_features_final = config["lyrics_features"]

spotify_weight = config["spotify_weight"]
lyrics_weight = config["lyrics_weight"]
pca_audio_weight = config["pca_audio_weight"]

spotify_scaler = StandardScaler()
lyrics_scaler = StandardScaler()
pca_audio_scaler = StandardScaler()

spotify_scaled = spotify_scaler.fit_transform(
    recommender_df[spotify_features]
)

lyrics_scaled = lyrics_scaler.fit_transform(
    recommender_df[lyrics_features_final]
)

pca_audio_scaled = pca_audio_scaler.fit_transform(
    pca10_audio_features
)

X_recommender = np.hstack([
    spotify_scaled * spotify_weight,
    lyrics_scaled * lyrics_weight,
    pca_audio_scaled * pca_audio_weight
])

print("Final recommender matrix shape:", X_recommender.shape)

Final recommender matrix shape: (71405, 26)


In [7]:
knn_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=51
)

knn_model.fit(X_recommender)

print("Optimized hybrid recommender fitted.")

Optimized hybrid recommender fitted.


## Track Search

Before recommendations can be generated, the user must be able to search for a song.

The search function returns matching tracks sorted by popularity, so the most recognizable version of a song appears first.

In [8]:
def search_tracks(query, max_results=10):
    matches = recommender_df[
        recommender_df["name"].str.contains(
            query,
            case=False,
            na=False
        )
    ].copy()

    matches = matches.sort_values(
        "popularity",
        ascending=False
    )

    display_cols = [
        "id",
        "name",
        "popularity",
        "eval_genres",
        "preview_url",
        "uri"
    ]

    return matches[display_cols].head(max_results)

In [9]:
search_tracks("Billie Jean", max_results=10)

,id,name,popularity,eval_genres,preview_url,uri
53749,5ChkMS8OtdzJeqyybCc9R5,Billie Jean,84.0,[pop],https://p.scdn.co/mp3-preview/4eb779428d40d579...,spotify:track:5ChkMS8OtdzJeqyybCc9R5
31267,4zzi2eD2cEPpQ3a307mPPj,Billie Jean,67.0,"[country, pop, other, folk_acoustic]",https://p.scdn.co/mp3-preview/a6a03eb031d4c2cb...,spotify:track:4zzi2eD2cEPpQ3a307mPPj
38967,6vR5u5b8JeRESx5nZaIWx6,Billie Jean,57.0,[pop],https://p.scdn.co/mp3-preview/f89e85eee3b4a703...,spotify:track:6vR5u5b8JeRESx5nZaIWx6
3260,7D0YmNlI9FDwodlKzHraRp,Billie Jean,47.0,[pop],https://p.scdn.co/mp3-preview/48f364b15efb0072...,spotify:track:7D0YmNlI9FDwodlKzHraRp
13700,6P3Rm624DzeUbCnkcbwj7B,Billie Jean - Live @ HMH - 27Jun09,32.0,"[rock, pop]",https://p.scdn.co/mp3-preview/4604426c6883f147...,spotify:track:6P3Rm624DzeUbCnkcbwj7B


## Initial Recommendation Function

The first recommendation function retrieves tracks that are closest to a selected query track in the optimized hybrid feature space.

This represents the first recommendation round before any user feedback is applied.

In [10]:
def recommend_by_index(
    query_idx,
    n_recommendations=10,
    remove_same_title=True
):
    query_title = recommender_df.loc[query_idx, "name"]

    distances, indices = knn_model.kneighbors(
        X_recommender[query_idx].reshape(1, -1),
        n_neighbors=n_recommendations + 30
    )

    recommendations = recommender_df.iloc[indices[0]].copy()
    recommendations["cosine_distance"] = distances[0]

    recommendations = recommendations[
        recommendations.index != query_idx
    ]

    if remove_same_title:
        recommendations = recommendations[
            recommendations["name"].str.lower()
            != query_title.lower()
        ]

    result_cols = [
        "id",
        "name",
        "popularity",
        "eval_genres",
        "preview_url",
        "uri",
        "cosine_distance"
    ]

    return recommendations[result_cols].head(n_recommendations)

In [11]:
billie_jean_idx = search_tracks("Billie Jean", max_results=1).index[0]

print("Query track:")
display(recommender_df.loc[[billie_jean_idx], ["name", "popularity", "eval_genres", "preview_url", "uri"]])

recommend_by_index(
    billie_jean_idx,
    n_recommendations=10
)

Query track:


,name,popularity,eval_genres,preview_url,uri
53749,Billie Jean,84.0,[pop],https://p.scdn.co/mp3-preview/4eb779428d40d579...,spotify:track:5ChkMS8OtdzJeqyybCc9R5


,id,name,popularity,eval_genres,preview_url,uri,cosine_distance
47192,6r05pfZigq7Arg25pi2vRm,Checklist (with Calvin Harris) (feat. WizKid),65.0,"[hip_hop, pop, other, electronic]",https://p.scdn.co/mp3-preview/e2a45820cfb1751a...,spotify:track:6r05pfZigq7Arg25pi2vRm,0.107406
53987,5xesHO6GLWTrjsKOjAXRmf,Checklist (with Calvin Harris) (feat. WizKid),64.0,"[hip_hop, pop, other, electronic]",https://p.scdn.co/mp3-preview/e2a45820cfb1751a...,spotify:track:5xesHO6GLWTrjsKOjAXRmf,0.111138
39163,2z4pcBLQXF2BXKFvd0BuB6,Tip Toe (feat. French Montana),76.0,"[hip_hop, pop, other]",https://p.scdn.co/mp3-preview/1e99ffb3917c9531...,spotify:track:2z4pcBLQXF2BXKFvd0BuB6,0.111454
48594,2RP8Svo0pMwZXnVcmOffDw,Fading,80.0,"[hip_hop, electronic]",https://p.scdn.co/mp3-preview/3b5c1bebdf2addfd...,spotify:track:2RP8Svo0pMwZXnVcmOffDw,0.111485
14237,6wmAHw1szh5RCKSRjiXhPe,How Long,79.0,[pop],https://p.scdn.co/mp3-preview/0dc00d9cdee8c916...,spotify:track:6wmAHw1szh5RCKSRjiXhPe,0.112304
27815,3oobWqIgYrSRg18uqRuPIF,Superstar,69.0,"[pop, electronic]",https://p.scdn.co/mp3-preview/84dc2fea13946ac6...,spotify:track:3oobWqIgYrSRg18uqRuPIF,0.119101
1123,1lsBTdE6MGsKeZCD6llNu7,Done for Me (feat. Kehlani),77.0,"[pop, other]",https://p.scdn.co/mp3-preview/cb26531c559b909f...,spotify:track:1lsBTdE6MGsKeZCD6llNu7,0.120211
54001,4c2W3VKsOFoIg2SFaO6DY5,Your Song,79.0,"[pop, other, electronic]",https://p.scdn.co/mp3-preview/91a31303c4101d8c...,spotify:track:4c2W3VKsOFoIg2SFaO6DY5,0.123626
48242,0KKkJNfGyhkQ5aFogxQAPU,That's What I Like,85.0,[pop],https://p.scdn.co/mp3-preview/f09116ce18bd77f5...,spotify:track:0KKkJNfGyhkQ5aFogxQAPU,0.126360
49846,2H9zpvV33I48HE5jnNvWVB,Fine Girl,69.0,[hip_hop],https://p.scdn.co/mp3-preview/e0072454c35ada6b...,spotify:track:2H9zpvV33I48HE5jnNvWVB,0.126610


In [17]:
def recommend_by_index(
    query_idx,
    n_recommendations=10,
    remove_same_title=True,
    remove_duplicate_titles=True
):
    query_title = recommender_df.loc[query_idx, "name"]

    distances, indices = knn_model.kneighbors(
        X_recommender[query_idx].reshape(1, -1),
        n_neighbors=n_recommendations + 50
    )

    recommendations = recommender_df.iloc[indices[0]].copy()
    recommendations["cosine_distance"] = distances[0]

    recommendations = recommendations[
        recommendations.index != query_idx
    ]

    if remove_same_title:
        recommendations = recommendations[
            recommendations["name"].str.lower()
            != query_title.lower()
        ]

    if remove_duplicate_titles:
        recommendations = recommendations.drop_duplicates(
            subset=["name"],
            keep="first"
        )

    result_cols = [
        "id",
        "name",
        "popularity",
        "eval_genres",
        "preview_url",
        "spotify_url",
        "cosine_distance"
    ]   

    return recommendations[result_cols].head(n_recommendations)

In [19]:
recommend_by_index(
    billie_jean_idx,
    n_recommendations=10
)

,id,name,popularity,eval_genres,preview_url,spotify_url,cosine_distance
47192,6r05pfZigq7Arg25pi2vRm,Checklist (with Calvin Harris) (feat. WizKid),65.0,"[hip_hop, pop, other, electronic]",https://p.scdn.co/mp3-preview/e2a45820cfb1751a...,https://open.spotify.com/track/6r05pfZigq7Arg2...,0.107406
39163,2z4pcBLQXF2BXKFvd0BuB6,Tip Toe (feat. French Montana),76.0,"[hip_hop, pop, other]",https://p.scdn.co/mp3-preview/1e99ffb3917c9531...,https://open.spotify.com/track/2z4pcBLQXF2BXKF...,0.111454
48594,2RP8Svo0pMwZXnVcmOffDw,Fading,80.0,"[hip_hop, electronic]",https://p.scdn.co/mp3-preview/3b5c1bebdf2addfd...,https://open.spotify.com/track/2RP8Svo0pMwZXnV...,0.111485
14237,6wmAHw1szh5RCKSRjiXhPe,How Long,79.0,[pop],https://p.scdn.co/mp3-preview/0dc00d9cdee8c916...,https://open.spotify.com/track/6wmAHw1szh5RCKS...,0.112304
27815,3oobWqIgYrSRg18uqRuPIF,Superstar,69.0,"[pop, electronic]",https://p.scdn.co/mp3-preview/84dc2fea13946ac6...,https://open.spotify.com/track/3oobWqIgYrSRg18...,0.119101
1123,1lsBTdE6MGsKeZCD6llNu7,Done for Me (feat. Kehlani),77.0,"[pop, other]",https://p.scdn.co/mp3-preview/cb26531c559b909f...,https://open.spotify.com/track/1lsBTdE6MGsKeZC...,0.120211
54001,4c2W3VKsOFoIg2SFaO6DY5,Your Song,79.0,"[pop, other, electronic]",https://p.scdn.co/mp3-preview/91a31303c4101d8c...,https://open.spotify.com/track/4c2W3VKsOFoIg2S...,0.123626
48242,0KKkJNfGyhkQ5aFogxQAPU,That's What I Like,85.0,[pop],https://p.scdn.co/mp3-preview/f09116ce18bd77f5...,https://open.spotify.com/track/0KKkJNfGyhkQ5aF...,0.126360
49846,2H9zpvV33I48HE5jnNvWVB,Fine Girl,69.0,[hip_hop],https://p.scdn.co/mp3-preview/e0072454c35ada6b...,https://open.spotify.com/track/2H9zpvV33I48HE5...,0.126610
47414,77YLeSEI8unMbZVWNUqioQ,Only Thing We Know,70.0,"[hip_hop, pop, electronic]",https://p.scdn.co/mp3-preview/bf6475670330cab9...,https://open.spotify.com/track/77YLeSEI8unMbZV...,0.127094


## Add Spotify link helper

In [20]:
def spotify_uri_to_url(uri):
    if pd.isna(uri):
        return None

    if isinstance(uri, str) and uri.startswith("spotify:track:"):
        track_id = uri.split(":")[-1]
        return f"https://open.spotify.com/track/{track_id}"

    return None


recommender_df["spotify_url"] = recommender_df["uri"].apply(spotify_uri_to_url)

recommender_df[["name", "uri", "spotify_url"]].head()

,name,uri,spotify_url
0,Already Gone,spotify:track:4PrAZpH9Ic7S47E78BN6E4,https://open.spotify.com/track/4PrAZpH9Ic7S47E...
1,Creature Kind,spotify:track:01zME4q62SDPtD0hOSmTrG,https://open.spotify.com/track/01zME4q62SDPtD0...
2,Greatest Comedian,spotify:track:2D0the6JBRyOpMeBkPFkGY,https://open.spotify.com/track/2D0the6JBRyOpMe...
3,I Wish I Was A Shark,spotify:track:0tewjlNbotxqF2obibsg36,https://open.spotify.com/track/0tewjlNbotxqF2o...
4,Cola Falls,spotify:track:2Dh5wED4UVeiBqneUdc5Gy,https://open.spotify.com/track/2Dh5wED4UVeiBqn...


In [21]:
result_cols = [
    "id",
    "name",
    "popularity",
    "eval_genres",
    "preview_url",
    "spotify_url",
    "cosine_distance"
]

In [22]:
recommend_by_index(
    billie_jean_idx,
    n_recommendations=10
)

,id,name,popularity,eval_genres,preview_url,spotify_url,cosine_distance
47192,6r05pfZigq7Arg25pi2vRm,Checklist (with Calvin Harris) (feat. WizKid),65.0,"[hip_hop, pop, other, electronic]",https://p.scdn.co/mp3-preview/e2a45820cfb1751a...,https://open.spotify.com/track/6r05pfZigq7Arg2...,0.107406
39163,2z4pcBLQXF2BXKFvd0BuB6,Tip Toe (feat. French Montana),76.0,"[hip_hop, pop, other]",https://p.scdn.co/mp3-preview/1e99ffb3917c9531...,https://open.spotify.com/track/2z4pcBLQXF2BXKF...,0.111454
48594,2RP8Svo0pMwZXnVcmOffDw,Fading,80.0,"[hip_hop, electronic]",https://p.scdn.co/mp3-preview/3b5c1bebdf2addfd...,https://open.spotify.com/track/2RP8Svo0pMwZXnV...,0.111485
14237,6wmAHw1szh5RCKSRjiXhPe,How Long,79.0,[pop],https://p.scdn.co/mp3-preview/0dc00d9cdee8c916...,https://open.spotify.com/track/6wmAHw1szh5RCKS...,0.112304
27815,3oobWqIgYrSRg18uqRuPIF,Superstar,69.0,"[pop, electronic]",https://p.scdn.co/mp3-preview/84dc2fea13946ac6...,https://open.spotify.com/track/3oobWqIgYrSRg18...,0.119101
1123,1lsBTdE6MGsKeZCD6llNu7,Done for Me (feat. Kehlani),77.0,"[pop, other]",https://p.scdn.co/mp3-preview/cb26531c559b909f...,https://open.spotify.com/track/1lsBTdE6MGsKeZC...,0.120211
54001,4c2W3VKsOFoIg2SFaO6DY5,Your Song,79.0,"[pop, other, electronic]",https://p.scdn.co/mp3-preview/91a31303c4101d8c...,https://open.spotify.com/track/4c2W3VKsOFoIg2S...,0.123626
48242,0KKkJNfGyhkQ5aFogxQAPU,That's What I Like,85.0,[pop],https://p.scdn.co/mp3-preview/f09116ce18bd77f5...,https://open.spotify.com/track/0KKkJNfGyhkQ5aF...,0.126360
49846,2H9zpvV33I48HE5jnNvWVB,Fine Girl,69.0,[hip_hop],https://p.scdn.co/mp3-preview/e0072454c35ada6b...,https://open.spotify.com/track/2H9zpvV33I48HE5...,0.126610
47414,77YLeSEI8unMbZVWNUqioQ,Only Thing We Know,70.0,"[hip_hop, pop, electronic]",https://p.scdn.co/mp3-preview/bf6475670330cab9...,https://open.spotify.com/track/77YLeSEI8unMbZV...,0.127094


## Simulating User Feedback

The first recommendation round is still purely content-based.

To make the recommender interactive, we now simulate user feedback. A user can like or dislike recommended tracks. These liked and disliked tracks are then used to build a temporary user preference profile.

In [23]:
initial_recommendations = recommend_by_index(
    billie_jean_idx,
    n_recommendations=10
)

initial_recommendations

,id,name,popularity,eval_genres,preview_url,spotify_url,cosine_distance
47192,6r05pfZigq7Arg25pi2vRm,Checklist (with Calvin Harris) (feat. WizKid),65.0,"[hip_hop, pop, other, electronic]",https://p.scdn.co/mp3-preview/e2a45820cfb1751a...,https://open.spotify.com/track/6r05pfZigq7Arg2...,0.107406
39163,2z4pcBLQXF2BXKFvd0BuB6,Tip Toe (feat. French Montana),76.0,"[hip_hop, pop, other]",https://p.scdn.co/mp3-preview/1e99ffb3917c9531...,https://open.spotify.com/track/2z4pcBLQXF2BXKF...,0.111454
48594,2RP8Svo0pMwZXnVcmOffDw,Fading,80.0,"[hip_hop, electronic]",https://p.scdn.co/mp3-preview/3b5c1bebdf2addfd...,https://open.spotify.com/track/2RP8Svo0pMwZXnV...,0.111485
14237,6wmAHw1szh5RCKSRjiXhPe,How Long,79.0,[pop],https://p.scdn.co/mp3-preview/0dc00d9cdee8c916...,https://open.spotify.com/track/6wmAHw1szh5RCKS...,0.112304
27815,3oobWqIgYrSRg18uqRuPIF,Superstar,69.0,"[pop, electronic]",https://p.scdn.co/mp3-preview/84dc2fea13946ac6...,https://open.spotify.com/track/3oobWqIgYrSRg18...,0.119101
1123,1lsBTdE6MGsKeZCD6llNu7,Done for Me (feat. Kehlani),77.0,"[pop, other]",https://p.scdn.co/mp3-preview/cb26531c559b909f...,https://open.spotify.com/track/1lsBTdE6MGsKeZC...,0.120211
54001,4c2W3VKsOFoIg2SFaO6DY5,Your Song,79.0,"[pop, other, electronic]",https://p.scdn.co/mp3-preview/91a31303c4101d8c...,https://open.spotify.com/track/4c2W3VKsOFoIg2S...,0.123626
48242,0KKkJNfGyhkQ5aFogxQAPU,That's What I Like,85.0,[pop],https://p.scdn.co/mp3-preview/f09116ce18bd77f5...,https://open.spotify.com/track/0KKkJNfGyhkQ5aF...,0.126360
49846,2H9zpvV33I48HE5jnNvWVB,Fine Girl,69.0,[hip_hop],https://p.scdn.co/mp3-preview/e0072454c35ada6b...,https://open.spotify.com/track/2H9zpvV33I48HE5...,0.126610
47414,77YLeSEI8unMbZVWNUqioQ,Only Thing We Know,70.0,"[hip_hop, pop, electronic]",https://p.scdn.co/mp3-preview/bf6475670330cab9...,https://open.spotify.com/track/77YLeSEI8unMbZV...,0.127094


In [24]:
liked_track_indices = [
    14237,  # How Long
    48242,  # That's What I Like
    1123    # Done for Me
]

disliked_track_indices = [
    49846,  # Fine Girl
    48594   # Fading
]

In [25]:
print("Liked tracks:")
display(recommender_df.loc[
    liked_track_indices,
    ["name", "popularity", "eval_genres", "spotify_url"]
])

print("Disliked tracks:")
display(recommender_df.loc[
    disliked_track_indices,
    ["name", "popularity", "eval_genres", "spotify_url"]
])

Liked tracks:


,name,popularity,eval_genres,spotify_url
14237,How Long,79.0,[pop],https://open.spotify.com/track/6wmAHw1szh5RCKS...
48242,That's What I Like,85.0,[pop],https://open.spotify.com/track/0KKkJNfGyhkQ5aF...
1123,Done for Me (feat. Kehlani),77.0,"[pop, other]",https://open.spotify.com/track/1lsBTdE6MGsKeZC...


Disliked tracks:


,name,popularity,eval_genres,spotify_url
49846,Fine Girl,69.0,[hip_hop],https://open.spotify.com/track/2H9zpvV33I48HE5...
48594,Fading,80.0,"[hip_hop, electronic]",https://open.spotify.com/track/2RP8Svo0pMwZXnV...


## Building a User Preference Profile

The liked and disliked tracks are converted into a temporary user profile.

The profile is calculated as:

liked track vectors average minus disliked track vectors average

This means the recommender moves closer to the songs the user liked and away from the songs the user disliked.

In [26]:
def build_user_profile(
    liked_indices,
    disliked_indices,
    liked_weight=1.0,
    disliked_weight=1.0
):
    liked_vectors = X_recommender[liked_indices]
    disliked_vectors = X_recommender[disliked_indices]

    liked_profile = liked_vectors.mean(axis=0)

    if len(disliked_indices) > 0:
        disliked_profile = disliked_vectors.mean(axis=0)
        user_profile = (liked_weight * liked_profile) - (disliked_weight * disliked_profile)
    else:
        user_profile = liked_weight * liked_profile

    return user_profile.reshape(1, -1)

In [27]:
def recommend_from_user_profile(
    liked_indices,
    disliked_indices,
    n_recommendations=10,
    exclude_seen=True
):
    user_profile = build_user_profile(
        liked_indices,
        disliked_indices
    )

    distances, indices = knn_model.kneighbors(
        user_profile,
        n_neighbors=n_recommendations + len(liked_indices) + len(disliked_indices) + 50
    )

    recommendations = recommender_df.iloc[indices[0]].copy()
    recommendations["cosine_distance"] = distances[0]

    if exclude_seen:
        seen_indices = set(liked_indices + disliked_indices)
        recommendations = recommendations[
            ~recommendations.index.isin(seen_indices)
        ]

    recommendations = recommendations.drop_duplicates(
        subset=["name"],
        keep="first"
    )

    result_cols = [
        "id",
        "name",
        "popularity",
        "eval_genres",
        "preview_url",
        "spotify_url",
        "cosine_distance"
    ]

    return recommendations[result_cols].head(n_recommendations)

In [28]:
personalized_recommendations = recommend_from_user_profile(
    liked_indices=liked_track_indices,
    disliked_indices=disliked_track_indices,
    n_recommendations=10
)

personalized_recommendations

,id,name,popularity,eval_genres,preview_url,spotify_url,cosine_distance
24579,5chgte6PLqxDVWXIUhVwBn,Caution,65.0,"[pop, other]",https://p.scdn.co/mp3-preview/23ebbe7216d4ba7c...,https://open.spotify.com/track/5chgte6PLqxDVWX...,0.285142
44303,4GuQY57gbi7DEgc2AJoyBh,Algo Oficial,43.0,"[latin, pop, other]",https://p.scdn.co/mp3-preview/8e025b7d35112c3e...,https://open.spotify.com/track/4GuQY57gbi7DEgc...,0.289880
60530,0jwGxgupGo2ovlU5eonF9D,Você Vai Ver,59.0,[pop],https://p.scdn.co/mp3-preview/ec313f0e5c252b9f...,https://open.spotify.com/track/0jwGxgupGo2ovlU...,0.305017
50244,11lScnLmiPDjqQ9SM6G0hj,Mas Alla Del Sol,62.0,"[latin, other]",https://p.scdn.co/mp3-preview/0fa40fa92aa66b24...,https://open.spotify.com/track/11lScnLmiPDjqQ9...,0.305856
43195,53orRX6MudB9jhW0sKZWhW,Creiste,52.0,"[latin, pop, other]",https://p.scdn.co/mp3-preview/bc3ab8fcc2eed2ff...,https://open.spotify.com/track/53orRX6MudB9jhW...,0.317486
39980,4WH2PelEpTVSuglBbz65gN,Somebody Special,62.0,"[pop, other]",https://p.scdn.co/mp3-preview/7369e470cfd615dc...,https://open.spotify.com/track/4WH2PelEpTVSugl...,0.320184
14800,2dlZfsBwCAtrzN5ETcCOZG,T'attends quoi,50.0,"[jazz, other]",https://p.scdn.co/mp3-preview/07ef71c64e51f8ba...,https://open.spotify.com/track/2dlZfsBwCAtrzN5...,0.325399
5385,1kNVJQEkobOlyfbctPZ4fs,Amar Amei,66.0,"[rnb_soul, electronic]",https://p.scdn.co/mp3-preview/d2f40c7fa10fc38b...,https://open.spotify.com/track/1kNVJQEkobOlyfb...,0.325460
18723,2t3ebknYxQR4ZJRqYA8qr1,Underdog,51.0,"[rock, pop]",https://p.scdn.co/mp3-preview/675a4f621dcd57ba...,https://open.spotify.com/track/2t3ebknYxQR4ZJR...,0.332622
6078,567sFGii0JsIoFlHkO6h8Z,Amigo (feat. Mula B),66.0,"[hip_hop, other]",https://p.scdn.co/mp3-preview/73851ba837ec1ec8...,https://open.spotify.com/track/567sFGii0JsIoFl...,0.341630


## Test lower dislike weight

In [29]:
def recommend_from_user_profile(
    liked_indices,
    disliked_indices,
    n_recommendations=10,
    liked_weight=1.0,
    disliked_weight=0.5,
    exclude_seen=True
):
    user_profile = build_user_profile(
        liked_indices,
        disliked_indices,
        liked_weight=liked_weight,
        disliked_weight=disliked_weight
    )

    distances, indices = knn_model.kneighbors(
        user_profile,
        n_neighbors=n_recommendations + len(liked_indices) + len(disliked_indices) + 50
    )

    recommendations = recommender_df.iloc[indices[0]].copy()
    recommendations["cosine_distance"] = distances[0]

    if exclude_seen:
        seen_indices = set(liked_indices + disliked_indices)
        recommendations = recommendations[
            ~recommendations.index.isin(seen_indices)
        ]

    recommendations = recommendations.drop_duplicates(
        subset=["name"],
        keep="first"
    )

    result_cols = [
        "id",
        "name",
        "popularity",
        "eval_genres",
        "preview_url",
        "spotify_url",
        "cosine_distance"
    ]

    return recommendations[result_cols].head(n_recommendations)

In [30]:
personalized_recommendations = recommend_from_user_profile(
    liked_indices=liked_track_indices,
    disliked_indices=disliked_track_indices,
    n_recommendations=10,
    disliked_weight=0.5
)

personalized_recommendations

,id,name,popularity,eval_genres,preview_url,spotify_url,cosine_distance
54001,4c2W3VKsOFoIg2SFaO6DY5,Your Song,79.0,"[pop, other, electronic]",https://p.scdn.co/mp3-preview/91a31303c4101d8c...,https://open.spotify.com/track/4c2W3VKsOFoIg2S...,0.100461
13196,7J41dYQolQJEtj3UmKLu5r,U Got It Bad,72.0,"[hip_hop, pop, other]",https://p.scdn.co/mp3-preview/d8ed27f510de0d7e...,https://open.spotify.com/track/7J41dYQolQJEtj3...,0.124538
44901,7iNiRaxUOvd4seviLprqeb,X,68.0,"[hip_hop, pop]",https://p.scdn.co/mp3-preview/37401199a1b2504e...,https://open.spotify.com/track/7iNiRaxUOvd4sev...,0.130273
45178,5cbpoIu3YjoOwbBDGUEp3P,Car Radio,76.0,"[rock, pop]",https://p.scdn.co/mp3-preview/ea3ed2e51bccacc2...,https://open.spotify.com/track/5cbpoIu3YjoOwbB...,0.131945
53277,6F8j6x7shbePnsC2SiAh04,Oh la folle,65.0,"[hip_hop, pop, other]",https://p.scdn.co/mp3-preview/328177bed45e1cd9...,https://open.spotify.com/track/6F8j6x7shbePnsC...,0.135572
48639,0PG9fbaaHFHfre2gUVo7AN,Please Me,93.0,"[hip_hop, pop]",https://p.scdn.co/mp3-preview/a25d43f3262e0762...,https://open.spotify.com/track/0PG9fbaaHFHfre2...,0.140716
49165,6C3MGasn7adPXHlE1z4NbV,FU4E,69.0,"[pop, electronic]",https://p.scdn.co/mp3-preview/9ec212bd6a7991ce...,https://open.spotify.com/track/6C3MGasn7adPXHl...,0.142047
9207,7gNlaZaMIjRUfnAYd9Dvmt,Aristocrate,72.0,[hip_hop],https://p.scdn.co/mp3-preview/38a2a03026081c7c...,https://open.spotify.com/track/7gNlaZaMIjRUfnA...,0.142115
53970,7BKLCZ1jbUBVqRi2FVlTVw,Closer,87.0,"[pop, electronic]",https://p.scdn.co/mp3-preview/8d3df1c64907cb18...,https://open.spotify.com/track/7BKLCZ1jbUBVqRi...,0.142306
27815,3oobWqIgYrSRg18uqRuPIF,Superstar,69.0,"[pop, electronic]",https://p.scdn.co/mp3-preview/84dc2fea13946ac6...,https://open.spotify.com/track/3oobWqIgYrSRg18...,0.142527


## Genre-Aware Feedback Filtering

The user profile already adapts to liked and disliked songs in the feature space. However, because some genres overlap strongly in audio and lyrics, the model can still recommend tracks from genres the user disliked.

To make the feedback behavior more intuitive, we add an optional genre-aware filter. It keeps recommendations that share at least one genre with the liked tracks.

In [31]:
def collect_genres(indices):
    genres = []

    for idx in indices:
        genres.extend(recommender_df.loc[idx, "eval_genres"])

    return set(genres)


liked_genres = collect_genres(liked_track_indices)
disliked_genres = collect_genres(disliked_track_indices)

print("Liked genres:", liked_genres)
print("Disliked genres:", disliked_genres)

Liked genres: {'pop', 'other'}
Disliked genres: {'hip_hop', 'electronic'}


In [32]:
def recommend_from_user_profile(
    liked_indices,
    disliked_indices,
    n_recommendations=10,
    liked_weight=1.0,
    disliked_weight=0.5,
    require_liked_genre=True,
    exclude_seen=True
):
    liked_genres = collect_genres(liked_indices)

    user_profile = build_user_profile(
        liked_indices,
        disliked_indices,
        liked_weight=liked_weight,
        disliked_weight=disliked_weight
    )

    distances, indices = knn_model.kneighbors(
        user_profile,
        n_neighbors=n_recommendations + len(liked_indices) + len(disliked_indices) + 200
    )

    recommendations = recommender_df.iloc[indices[0]].copy()
    recommendations["cosine_distance"] = distances[0]

    if exclude_seen:
        seen_indices = set(liked_indices + disliked_indices)
        recommendations = recommendations[
            ~recommendations.index.isin(seen_indices)
        ]

    if require_liked_genre:
        recommendations = recommendations[
            recommendations["eval_genres"].apply(
                lambda genres: len(set(genres) & liked_genres) > 0
            )
        ]

    recommendations = recommendations.drop_duplicates(
        subset=["name"],
        keep="first"
    )

    result_cols = [
        "id",
        "name",
        "popularity",
        "eval_genres",
        "preview_url",
        "spotify_url",
        "cosine_distance"
    ]

    return recommendations[result_cols].head(n_recommendations)

In [33]:
personalized_recommendations_genre_filtered = recommend_from_user_profile(
    liked_indices=liked_track_indices,
    disliked_indices=disliked_track_indices,
    n_recommendations=10,
    disliked_weight=0.5,
    require_liked_genre=True
)

personalized_recommendations_genre_filtered

,id,name,popularity,eval_genres,preview_url,spotify_url,cosine_distance
54001,4c2W3VKsOFoIg2SFaO6DY5,Your Song,79.0,"[pop, other, electronic]",https://p.scdn.co/mp3-preview/91a31303c4101d8c...,https://open.spotify.com/track/4c2W3VKsOFoIg2S...,0.100461
13196,7J41dYQolQJEtj3UmKLu5r,U Got It Bad,72.0,"[hip_hop, pop, other]",https://p.scdn.co/mp3-preview/d8ed27f510de0d7e...,https://open.spotify.com/track/7J41dYQolQJEtj3...,0.124538
44901,7iNiRaxUOvd4seviLprqeb,X,68.0,"[hip_hop, pop]",https://p.scdn.co/mp3-preview/37401199a1b2504e...,https://open.spotify.com/track/7iNiRaxUOvd4sev...,0.130273
45178,5cbpoIu3YjoOwbBDGUEp3P,Car Radio,76.0,"[rock, pop]",https://p.scdn.co/mp3-preview/ea3ed2e51bccacc2...,https://open.spotify.com/track/5cbpoIu3YjoOwbB...,0.131945
53277,6F8j6x7shbePnsC2SiAh04,Oh la folle,65.0,"[hip_hop, pop, other]",https://p.scdn.co/mp3-preview/328177bed45e1cd9...,https://open.spotify.com/track/6F8j6x7shbePnsC...,0.135572
48639,0PG9fbaaHFHfre2gUVo7AN,Please Me,93.0,"[hip_hop, pop]",https://p.scdn.co/mp3-preview/a25d43f3262e0762...,https://open.spotify.com/track/0PG9fbaaHFHfre2...,0.140716
49165,6C3MGasn7adPXHlE1z4NbV,FU4E,69.0,"[pop, electronic]",https://p.scdn.co/mp3-preview/9ec212bd6a7991ce...,https://open.spotify.com/track/6C3MGasn7adPXHl...,0.142047
53970,7BKLCZ1jbUBVqRi2FVlTVw,Closer,87.0,"[pop, electronic]",https://p.scdn.co/mp3-preview/8d3df1c64907cb18...,https://open.spotify.com/track/7BKLCZ1jbUBVqRi...,0.142306
27815,3oobWqIgYrSRg18uqRuPIF,Superstar,69.0,"[pop, electronic]",https://p.scdn.co/mp3-preview/84dc2fea13946ac6...,https://open.spotify.com/track/3oobWqIgYrSRg18...,0.142527
44886,4KoecuyOpZaNFZ0UqVsllc,Follow Me,72.0,"[rock, pop, country]",https://p.scdn.co/mp3-preview/482cf9985443c2e4...,https://open.spotify.com/track/4KoecuyOpZaNFZ0...,0.142709


## Stronger Genre-Aware Filtering

In [36]:
def recommend_from_user_profile(
    liked_indices,
    disliked_indices,
    seed_indices=None,
    n_recommendations=10,
    liked_weight=1.0,
    disliked_weight=0.5,
    require_liked_genre=True,
    remove_disliked_genre=False,
    exclude_seen=True
):
    if seed_indices is None:
        seed_indices = []

    liked_genres = collect_genres(liked_indices)
    disliked_genres = collect_genres(disliked_indices)

    user_profile = build_user_profile(
        liked_indices,
        disliked_indices,
        liked_weight=liked_weight,
        disliked_weight=disliked_weight
    )

    distances, indices = knn_model.kneighbors(
        user_profile,
        n_neighbors=n_recommendations + len(liked_indices) + len(disliked_indices) + len(seed_indices) + 300
    )

    recommendations = recommender_df.iloc[indices[0]].copy()
    recommendations["cosine_distance"] = distances[0]

    if exclude_seen:
        seen_indices = set(liked_indices + disliked_indices + seed_indices)
        recommendations = recommendations[
            ~recommendations.index.isin(seen_indices)
        ]

    if require_liked_genre:
        recommendations = recommendations[
            recommendations["eval_genres"].apply(
                lambda genres: len(set(genres) & liked_genres) > 0
            )
        ]

    if remove_disliked_genre:
        recommendations = recommendations[
            recommendations["eval_genres"].apply(
                lambda genres: len(set(genres) & disliked_genres) == 0
            )
        ]

    recommendations = recommendations.drop_duplicates(
        subset=["name"],
        keep="first"
    )

    result_cols = [
        "id",
        "name",
        "popularity",
        "eval_genres",
        "preview_url",
        "spotify_url",
        "cosine_distance"
    ]

    return recommendations[result_cols].head(n_recommendations)

In [37]:
personalized_recommendations_strict = recommend_from_user_profile(
    liked_indices=liked_track_indices,
    disliked_indices=disliked_track_indices,
    seed_indices=[billie_jean_idx],
    n_recommendations=10,
    disliked_weight=0.5,
    require_liked_genre=True,
    remove_disliked_genre=True
)

personalized_recommendations_strict

,id,name,popularity,eval_genres,preview_url,spotify_url,cosine_distance
45178,5cbpoIu3YjoOwbBDGUEp3P,Car Radio,76.0,"[rock, pop]",https://p.scdn.co/mp3-preview/ea3ed2e51bccacc2...,https://open.spotify.com/track/5cbpoIu3YjoOwbB...,0.131945
44886,4KoecuyOpZaNFZ0UqVsllc,Follow Me,72.0,"[rock, pop, country]",https://p.scdn.co/mp3-preview/482cf9985443c2e4...,https://open.spotify.com/track/4KoecuyOpZaNFZ0...,0.142709
48090,32OlwWuMpZ6b0aN2RZOeMS,Uptown Funk,82.0,[pop],https://p.scdn.co/mp3-preview/88a794d97fcb4475...,https://open.spotify.com/track/32OlwWuMpZ6b0aN...,0.146299
50315,6XiSceaDaO2vaYF2B2A70t,No Me Hubiera Enamorado,71.0,[pop],https://p.scdn.co/mp3-preview/b6231c150e81ede8...,https://open.spotify.com/track/6XiSceaDaO2vaYF...,0.146828
34461,6rsm9NTgl9kKPatf7S1yCS,Talk,71.0,"[pop, other]",https://p.scdn.co/mp3-preview/94ab807b008eb502...,https://open.spotify.com/track/6rsm9NTgl9kKPat...,0.150624
44753,3MLtopC0uho28PxZN7Zecy,Fake You Out,67.0,"[rock, pop]",https://p.scdn.co/mp3-preview/35e8b2da4c26dbfa...,https://open.spotify.com/track/3MLtopC0uho28Px...,0.152390
24579,5chgte6PLqxDVWXIUhVwBn,Caution,65.0,"[pop, other]",https://p.scdn.co/mp3-preview/23ebbe7216d4ba7c...,https://open.spotify.com/track/5chgte6PLqxDVWX...,0.157313
40844,7k6IzwMGpxnRghE7YosnXT,Me & U,73.0,"[pop, other]",https://p.scdn.co/mp3-preview/af06bccc66a9062a...,https://open.spotify.com/track/7k6IzwMGpxnRghE...,0.161276
43445,3c0uKKorZypy13KkO1u8AD,LBD,72.0,"[latin, pop]",https://p.scdn.co/mp3-preview/fb96858a9423584b...,https://open.spotify.com/track/3c0uKKorZypy13K...,0.162555
2222,1ok4gxOv8cg5WLjWK6TQhD,Go Go,70.0,[pop],https://p.scdn.co/mp3-preview/a94bba92d68e8b67...,https://open.spotify.com/track/1ok4gxOv8cg5WLj...,0.165825


## Feedback Behavior

The recommender uses disliked tracks as a negative signal in the feature space, but it does not automatically remove every genre that appears in a disliked song.

This is important because disliking one hip-hop song does not necessarily mean the user dislikes all hip-hop songs. Therefore, genre removal is treated as an optional stricter mode rather than the default recommendation behavior.

In [38]:
personalized_recommendations = recommend_from_user_profile(
    liked_indices=liked_track_indices,
    disliked_indices=disliked_track_indices,
    seed_indices=[billie_jean_idx],
    n_recommendations=10,
    disliked_weight=0.5,
    require_liked_genre=True,
    remove_disliked_genre=False
)

personalized_recommendations

,id,name,popularity,eval_genres,preview_url,spotify_url,cosine_distance
54001,4c2W3VKsOFoIg2SFaO6DY5,Your Song,79.0,"[pop, other, electronic]",https://p.scdn.co/mp3-preview/91a31303c4101d8c...,https://open.spotify.com/track/4c2W3VKsOFoIg2S...,0.100461
13196,7J41dYQolQJEtj3UmKLu5r,U Got It Bad,72.0,"[hip_hop, pop, other]",https://p.scdn.co/mp3-preview/d8ed27f510de0d7e...,https://open.spotify.com/track/7J41dYQolQJEtj3...,0.124538
44901,7iNiRaxUOvd4seviLprqeb,X,68.0,"[hip_hop, pop]",https://p.scdn.co/mp3-preview/37401199a1b2504e...,https://open.spotify.com/track/7iNiRaxUOvd4sev...,0.130273
45178,5cbpoIu3YjoOwbBDGUEp3P,Car Radio,76.0,"[rock, pop]",https://p.scdn.co/mp3-preview/ea3ed2e51bccacc2...,https://open.spotify.com/track/5cbpoIu3YjoOwbB...,0.131945
53277,6F8j6x7shbePnsC2SiAh04,Oh la folle,65.0,"[hip_hop, pop, other]",https://p.scdn.co/mp3-preview/328177bed45e1cd9...,https://open.spotify.com/track/6F8j6x7shbePnsC...,0.135572
48639,0PG9fbaaHFHfre2gUVo7AN,Please Me,93.0,"[hip_hop, pop]",https://p.scdn.co/mp3-preview/a25d43f3262e0762...,https://open.spotify.com/track/0PG9fbaaHFHfre2...,0.140716
49165,6C3MGasn7adPXHlE1z4NbV,FU4E,69.0,"[pop, electronic]",https://p.scdn.co/mp3-preview/9ec212bd6a7991ce...,https://open.spotify.com/track/6C3MGasn7adPXHl...,0.142047
53970,7BKLCZ1jbUBVqRi2FVlTVw,Closer,87.0,"[pop, electronic]",https://p.scdn.co/mp3-preview/8d3df1c64907cb18...,https://open.spotify.com/track/7BKLCZ1jbUBVqRi...,0.142306
27815,3oobWqIgYrSRg18uqRuPIF,Superstar,69.0,"[pop, electronic]",https://p.scdn.co/mp3-preview/84dc2fea13946ac6...,https://open.spotify.com/track/3oobWqIgYrSRg18...,0.142527
44886,4KoecuyOpZaNFZ0UqVsllc,Follow Me,72.0,"[rock, pop, country]",https://p.scdn.co/mp3-preview/482cf9985443c2e4...,https://open.spotify.com/track/4KoecuyOpZaNFZ0...,0.142709


In [39]:
personalized_recommendations_strict = recommend_from_user_profile(
    liked_indices=liked_track_indices,
    disliked_indices=disliked_track_indices,
    seed_indices=[billie_jean_idx],
    n_recommendations=10,
    disliked_weight=0.5,
    require_liked_genre=True,
    remove_disliked_genre=True
)

personalized_recommendations_strict

,id,name,popularity,eval_genres,preview_url,spotify_url,cosine_distance
45178,5cbpoIu3YjoOwbBDGUEp3P,Car Radio,76.0,"[rock, pop]",https://p.scdn.co/mp3-preview/ea3ed2e51bccacc2...,https://open.spotify.com/track/5cbpoIu3YjoOwbB...,0.131945
44886,4KoecuyOpZaNFZ0UqVsllc,Follow Me,72.0,"[rock, pop, country]",https://p.scdn.co/mp3-preview/482cf9985443c2e4...,https://open.spotify.com/track/4KoecuyOpZaNFZ0...,0.142709
48090,32OlwWuMpZ6b0aN2RZOeMS,Uptown Funk,82.0,[pop],https://p.scdn.co/mp3-preview/88a794d97fcb4475...,https://open.spotify.com/track/32OlwWuMpZ6b0aN...,0.146299
50315,6XiSceaDaO2vaYF2B2A70t,No Me Hubiera Enamorado,71.0,[pop],https://p.scdn.co/mp3-preview/b6231c150e81ede8...,https://open.spotify.com/track/6XiSceaDaO2vaYF...,0.146828
34461,6rsm9NTgl9kKPatf7S1yCS,Talk,71.0,"[pop, other]",https://p.scdn.co/mp3-preview/94ab807b008eb502...,https://open.spotify.com/track/6rsm9NTgl9kKPat...,0.150624
44753,3MLtopC0uho28PxZN7Zecy,Fake You Out,67.0,"[rock, pop]",https://p.scdn.co/mp3-preview/35e8b2da4c26dbfa...,https://open.spotify.com/track/3MLtopC0uho28Px...,0.152390
24579,5chgte6PLqxDVWXIUhVwBn,Caution,65.0,"[pop, other]",https://p.scdn.co/mp3-preview/23ebbe7216d4ba7c...,https://open.spotify.com/track/5chgte6PLqxDVWX...,0.157313
40844,7k6IzwMGpxnRghE7YosnXT,Me & U,73.0,"[pop, other]",https://p.scdn.co/mp3-preview/af06bccc66a9062a...,https://open.spotify.com/track/7k6IzwMGpxnRghE...,0.161276
43445,3c0uKKorZypy13KkO1u8AD,LBD,72.0,"[latin, pop]",https://p.scdn.co/mp3-preview/fb96858a9423584b...,https://open.spotify.com/track/3c0uKKorZypy13K...,0.162555
2222,1ok4gxOv8cg5WLjWK6TQhD,Go Go,70.0,[pop],https://p.scdn.co/mp3-preview/a94bba92d68e8b67...,https://open.spotify.com/track/1ok4gxOv8cg5WLj...,0.165825


## Comparing Initial and Personalized Recommendations

The initial recommendations are based only on similarity to the selected seed song.

The personalized recommendations use the user's feedback to adjust the recommendation direction. This allows the system to move closer to liked tracks while reducing the influence of disliked tracks.

In [40]:
comparison_summary = pd.DataFrame({
    "Stage": [
        "Initial recommendations",
        "Personalized recommendations",
        "Strict genre-filtered recommendations"
    ],
    "Description": [
        "Based only on the selected seed song.",
        "Uses liked and disliked tracks to build a user preference profile.",
        "Additionally removes tracks containing genres from disliked songs."
    ],
    "Recommended Genres": [
        initial_recommendations["eval_genres"].explode().value_counts().to_dict(),
        personalized_recommendations["eval_genres"].explode().value_counts().to_dict(),
        personalized_recommendations_strict["eval_genres"].explode().value_counts().to_dict()
    ],
    "Average Popularity": [
        initial_recommendations["popularity"].mean(),
        personalized_recommendations["popularity"].mean(),
        personalized_recommendations_strict["popularity"].mean()
    ],
    "Average Distance": [
        initial_recommendations["cosine_distance"].mean(),
        personalized_recommendations["cosine_distance"].mean(),
        personalized_recommendations_strict["cosine_distance"].mean()
    ]
})

comparison_summary

,Stage,Description,Recommended Genres,Average Popularity,Average Distance
0,Initial recommendations,Based only on the selected seed song.,"{'pop': 8, 'hip_hop': 5, 'electronic': 5, 'oth...",74.9,0.118565
1,Personalized recommendations,Uses liked and disliked tracks to build a user...,"{'pop': 10, 'electronic': 4, 'hip_hop': 4, 'ot...",75.0,0.133309
2,Strict genre-filtered recommendations,Additionally removes tracks containing genres ...,"{'pop': 10, 'rock': 3, 'other': 3, 'country': ...",71.9,0.151776


## Notebook Outcome

This notebook demonstrates how the optimized content-based recommender can be extended with user feedback.

The system remains content-based because recommendations are generated from song features, but it becomes interactive by updating the recommendation profile based on likes and dislikes.

The final dashboard can now use this logic directly:

1. Search for a song
2. Show initial recommendations
3. Let the user like or dislike tracks
4. Build a temporary user profile
5. Generate personalized recommendations
6. Optionally apply stricter genre filtering